# RLHF Clinical Red-Teaming — CLI Driver
**Audrey Tjokro · Stephen Dong · Niki Karanikola**

This notebook is a **driver only**: no training logic, no hyperparameters, no model code lives here. It clones the repo at a pinned commit, installs deps, sets secrets, and shells out to `python -m src ...`.

Edit the **CONFIG** cell to choose method (`baseline` / `dpo` / `ppo`) and any overrides; everything else can be "Run all".

Per-run artifacts sync to `gs://results_043026/<method>/<run-uuid>/` at the end of the run.

## 1. Mount Drive (optional — used only as a local cache for HF + checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo at a specific commit/branch
Pin a SHA for paper-grade reproducibility. Pass `BRANCH = 'main'` for HEAD.

In [ ]:
REPO_URL = 'https://github.com/<your-org>/rlhf-clinical-redteaming.git'
BRANCH   = 'main'         # or a SHA like 'd07626a'
REPO_DIR = '/content/rlhf-clinical-redteaming'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())
%cd $REPO_DIR

## 3. Install requirements

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

## 4. Secrets + GCS auth
Add `OPENAI_API_KEY` to Colab Secrets (left sidebar → key icon).

In [ ]:
from google.colab import userdata, auth
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()

# GCS auth — uses your Google account; bucket gs://results_043026 must be in project rlhf-clinical-redteaming.
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'rlhf-clinical-redteaming'

## 5. CONFIG — pick method, run name, any overrides

In [ ]:
METHOD     = 'baseline'                # 'baseline' | 'dpo' | 'ppo'
CONFIG     = f'configs/{METHOD}.yaml'
RUN_NAME   = f'{METHOD}-colab'
GCS_BUCKET = 'gs://results_043026'

# Optional dotted-path overrides — same format as --override on the CLI.
OVERRIDES = []                          # e.g. ['data.n_test=20', 'max_turns=3']
EXTRA_FLAGS = []                        # e.g. ['--use-test'], ['--allow-dirty']

import shlex
cmd = ['python', '-m', 'src', METHOD,
       '--config', CONFIG,
       '--run-name', RUN_NAME,
       '--gcs-bucket', GCS_BUCKET]
for o in OVERRIDES:
    cmd += ['--override', o]
cmd += EXTRA_FLAGS
print('$', ' '.join(shlex.quote(c) for c in cmd))

## 6. Run the CLI (artifacts sync to GCS at the end automatically)

In [ ]:
import subprocess, sys
result = subprocess.run(cmd)
sys.exit(result.returncode) if result.returncode else print('OK')

## 7. (Optional) Inspect this run's artifacts in GCS

In [ ]:
!gsutil ls -r {GCS_BUCKET}/{METHOD}/ | tail -30